In [1]:
#Assignment 8: Gabriel Maldonado

In [2]:
import pygame
import moderngl
import numpy
import math
import glm
import assimp_py
from loadModelUsingAssimp_V2 import create3DAssimpObject

pygame.init()

width = 840
height = 480
pygame.display.gl_set_attribute(pygame.GL_CONTEXT_PROFILE_MASK, pygame.GL_CONTEXT_PROFILE_CORE)
pygame.display.gl_set_attribute(pygame.GL_MULTISAMPLESAMPLES, 16)
display = pygame.display.set_mode((width, height), flags= pygame.OPENGL | pygame.DOUBLEBUF | pygame.RESIZABLE)
display = pygame.display.set_caption(title = "Assignment 8 - Gabriel Maldonado")
gl = moderngl.get_context()
gl.info["GL_VERSION"]
#print(gl.max_samples) #16

C:\Users\vanes\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


'4.6.0 - Build 31.0.101.5186'

In [3]:
texture_vertex_shader_code = '''
    #version 330 core

    layout(location=0) in vec3 positions;
    layout(location=1) in vec3 in_normal;
    layout(location=2) in vec2 inUV;
    out vec2 uv;
    out vec3 normal;
    uniform mat4 model;
    uniform mat4 viewMatrix;
    uniform mat4 perspectiveMatrix;

    void main(){
        vec4 position = model * vec4(positions, 1);
        gl_Position = perspectiveMatrix * viewMatrix * position;
        uv = inUV;
        mat3 normalMatrix = mat3(transpose(inverse(model)));
        normal = normalize(normalMatrix * in_normal);
}

'''

In [4]:
texture_fragment_shader_code = '''
    #version 330 core

    in vec3 normal;
    in vec2 uv;
    out vec4 tColor;
    uniform sampler2D map;
    uniform vec3 light;

    vec3 computeColor(){
    vec3 L = normalize(light);
    vec3 N = normalize(normal);
    vec3 materialColor = texture(map, uv).rgb;
    return materialColor * clamp(dot(L,N), 0, 1);
}

    void main(){
        vec3 color = computeColor();
        tColor = vec4(color, 1);
}    
'''

In [5]:
program = gl.program(
    vertex_shader = texture_vertex_shader_code,
    fragment_shader = texture_fragment_shader_code
)

assimpObject = create3DAssimpObject("rayman/scene.gltf", verbose = False, textureFlag = True, normalFlag = True)

Accumulates Normals in the geomDataList
Accumulates Textures in the geomDataList
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.


In [6]:
#format = "3f 2f" 
#variables = ["positions", "inUV"]

format = "3f 3f 2f"
variables = ["positions", "in_normal", "inUV"]
#when I run the 2 lines above instead of the first two lines in this cell it creates a mess that does
#not look like the required object but have the lighting required however when I run the first two lines
#in this cell the object is rendered however the lighting is not

renderables = assimpObject.getRenderables(gl, program, format, variables)
samplers = assimpObject.getSamplers(gl)
bounds = assimpObject.bound
gl.enable(gl.DEPTH_TEST)
pause = False
running = True
clock = pygame.time.Clock()
camera_elevation_angle = 30
camera_y_angle = 0
camera_angle_in_radians  = math.radians(90 - camera_elevation_angle)
distance = 2 * bounds.radius
displacement_vector = glm.vec3(0, distance * math.cos(camera_angle_in_radians), distance * math.sin(camera_angle_in_radians))
target_point = glm.vec3(bounds.center)
up_vector = glm.vec3(0,1,0)


In [ ]:


while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif (event.type == pygame.KEYDOWN):
            if event.key == 27:
                running = False
            elif event.key == pygame.K_p:
                pause = not pause
        elif (event.type == pygame.WINDOWRESIZED):
            width = event.x
            height = event.y

    aspect = width/height
    new_displacement_vector = glm.rotate(displacement_vector, glm.radians(camera_y_angle), glm.vec3(0,1,0))

    #eye_y = distance * math.cos(camera_angle_in_radians)

    #eye_radius = distance * math.sin(camera_angle_in_radians)
    #eye_x = eye_radius * math.cos(math.radians(camera_angle_in_radians))
    #eye_z = eye_radius * math.sin(math.radians(camera_angle_in_radians))

    #eye_point = glm.vec3(bounds.center) + glm.rotate(displacement_vector, camera_y_angle, glm.vec3(0,1,0))
    eye_point = target_point + new_displacement_vector
    viewMatrix = glm.lookAt(eye_point, target_point, up_vector)    

    camera_perspective_matrix = glm.perspective(math.radians(45), aspect, bounds.radius, 3 * bounds.radius)         
            
    gl.clear(0.5,0.5,0)

    model = glm.mat4(1)
    
    program["light"] = new_displacement_vector
    program["viewMatrix"].write(viewMatrix)
    program["perspectiveMatrix"].write(camera_perspective_matrix)
    #program["model"].write(model)
    assimpObject.render(program, renderables, samplers, M = model)
    #render()

    
    pygame.display.flip()
    if not pause:
        clock.tick(30)
        camera_y_angle = camera_y_angle + 1
        if camera_y_angle > 360:
            camera_y_angle = 0
    
pygame.display.quit()


In [ ]:
renderables

In [ ]:
samplers